<img src="https://radar.community.uaf.edu/wp-content/uploads/sites/667/2021/03/HydroSARbanner.jpg" width="100%" />

<br>
<font size="6"> <b>FIER Daily Flood Forecasting Training Code</b><img style="padding: 7px" src="https://radar.community.uaf.edu/wp-content/uploads/sites/667/2021/03/UAFLogo_A_647.png" width="170" align="right"/></font>

<br>
<font size="4"> <b> Franz J Meyer, University of Alaska Fairbanks</b> <br>
</font>

This notebook produces polynomial functions and neural network models for use in forecasting daily flood inundation. The flood inundation predictions use time series of Sentinel-1 RTC data and GEOGLoWs river runoff forecasts. 
    
The workflow utilizes information available in the fierpy <a href="https://github.com/SERVIR/fierpy">fierpy</a> GitHub repository.
<hr>


# Load Python Libraries

In [ ]:
# installs to make Victor's code work
# These only need to be installed once. 
# !pip install --upgrade xeofs
# !python -m pip install "tensorflow==2.17.0"

In [ ]:
import xeofs as xe
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import glob
import re
import rioxarray as rxr
import rasterio
from pyproj import Transformer
import time
from collections import Counter
from fier_new_geoglows import get_streamflow as g_sf_new
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.layers import Normalization

print(tf.__version__)
from fier_local import *

In [ ]:
# import geoglows

## CREATE FUNCTIONS

In [ ]:
### --- Get Dates from filenames --- ###
def get_dates(dir_path, prefix):
    dates = []
    pths = list(dir_path.glob(f'{prefix}.tif*'))

    for p in pths:
        date_regex = r'\d{8}'
        date = re.search(date_regex, str(p))
        if date:
            dates.append(date.group(0))
    return dates





### --- Load Geotiffs Function --- ###
def load_tiffs(parent, type_file, pola, stop_ind = -1):

    # Load the appropriate files
    if type_file == 'SAR':
        folder = parent+'RTC_GAMMA/'
        prefix = f'*{pola}'
    else:
        folder = parent+'Water_Masks/'
        prefix = '*combined'

    # Gather names of files corresponding to the file type and polarization we want
    tiff_dir = Path(folder)
    tiffs = list(tiff_dir.glob(f'{prefix}.tif*'))
    
    # Gather the date of each file
    times = get_dates(tiff_dir, prefix)
    
    # Create a list of indices based on the sorted order of times
    sorted_indices = sorted(range(len(times)), key=lambda i: times[i])
    
    # Sort the paths based on the times
    tiffs = [tiffs[i] for i in sorted_indices]
    
    # Sort the times axisdd
    times.sort()
    times = pd.DatetimeIndex(times)
    times.name = "time"
    
    # Create the dataset gathering the input images
    da = xr.concat([rxr.open_rasterio(f).squeeze(dim='band') for f in tiffs[:stop_ind]], dim=times[:stop_ind])
    da = da.drop_vars(['band', 'spatial_ref'])

    print(f"Dataset size in memory: {da.nbytes / 1e6:.2f} MB")

    return da, tiffs, times

 



### --- Get boundary coordinates of geotiff --- ###       
def get_coordinates_from_geotiff_bbox(geotiff_path):
    dataset = gdal.Open(geotiff_path)

    # Get the GeoTransform to convert pixel coordinates to geographical coordinates
    transform = dataset.GetGeoTransform()

    # Get the raster size (number of rows and columns)
    cols = dataset.RasterXSize
    rows = dataset.RasterYSize

    # Get the four corner points in pixel coordinates
    corners = [(0, 0), (cols, 0), (cols, rows), (0, rows)]

    # Calculate the longitude (X) and latitude (Y) of each corner
    corner_coordinates = []
    for corner in corners:
        lon = transform[0] + corner[0] * transform[1] + corner[1] * transform[2]
        lat = transform[3] + corner[0] * transform[4] + corner[1] * transform[5]

        # Get the spatial reference system of the dataset
        srs = osr.SpatialReference()
        srs.ImportFromWkt(dataset.GetProjection())

        # Create a coordinate transformation object
        target_srs = osr.SpatialReference()
        target_srs.ImportFromEPSG(4326)  # EPSG code for WGS84
        transform_obj = osr.CoordinateTransformation(srs, target_srs)

        # Transform the coordinates to WGS84 (latitude and longitude)
        lon, lat, _ = transform_obj.TransformPoint(lon, lat)

        corner_coordinates.append((lat, lon))
        
    corner_coordinates = np.array(corner_coordinates)
    boundaries = np.array([[np.min(corner_coordinates[:,1]), np.min(corner_coordinates[:,0])],
                           [np.max(corner_coordinates[:,1]), np.max(corner_coordinates[:,0])]])

    return boundaries





### --- Get center coordinates of geotiff --- ###
def get_centerpoint_coordinates(tif_file):
    # Open the raster file
    with rasterio.open(tif_file) as dataset:
        # Get the CRS of the raster
        crs = dataset.crs
    
        # Get the bounds of the raster (left, bottom, right, top)
        bounds = dataset.bounds
    
        # Calculate the center point in the raster's CRS
        center_x = (bounds.left + bounds.right) / 2
        center_y = (bounds.bottom + bounds.top) / 2
    
        # Initialize the transformer to convert from the raster's CRS to EPSG:4326
        transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    
        # Transform the center point coordinates to EPSG:4326
        center_lon, center_lat = transformer.transform(center_x, center_y)
    
        print(f"Center Longitude: {center_lon}, Center Latitude: {center_lat}")

        return center_lat, center_lon





### --- REOF Analysis of dataset --- ###
def reof(da, n_modes, rtol):

    start_time = time.time()
    model = xe.single.EOF(n_modes=n_modes, standardize=True, use_coslat=False)
    model.fit(da, dim="time")
    # (2) Varimax-rotated EOF analysis
    rot_var = xe.single.EOFRotator(n_modes=n_modes, power=1, rtol = rtol)
    rot_var.fit(model)
    expvar = model.explained_variance()
    expvar_ratio = model.explained_variance_ratio() * 100
    pcs = rot_var.scores()
    modes = rot_var.components()    

    # End time
    end_time = time.time()
    
    # Calculate the elapsed time
    elapsed_time = end_time - start_time
    print(f"REOF time: {elapsed_time:.2f} seconds")

    # Rename the DataArrays
    spatial_modes = modes.rename('spatial_modes')
    temporal_modes = pcs.rename('temporal_modes')
    
    # Flatten data from [t, y, x] to [t, space]
    da_flat = da.stack(space=('y', 'x'))
    
    # Compute the temporal mean for each pixel and reshape
    center = da_flat.mean(dim='time').unstack('space')
    
    # Create a DataArray for 'center' with 'y' and 'x'
    center = xr.DataArray(center, dims=['y', 'x'], coords={'y': modes.y, 'x': modes.x}, name='center')
    
    # Merge into a single dataset
    reof_ds = xr.Dataset({'spatial_modes': spatial_modes, 'temporal_modes': temporal_modes, 'center': center})

    return reof_ds 





# --- Polynomial fits of modes for discharge --- ###
def best_polynomial_fits(reof_ds, q, max_deg, NSE_cut, myParent=None, s_f=None, figSize=None, max_modes=None, myAxes=None):

    # Generate list of degrees from 1 to max_deg
    degrees = np.arange(1, max_deg+1)

    # Initialize storage lists
    tot_score = []
    tot_mode = []
    tot_deg = []
    tot_poly = []
    best_score = []

    # select modes to use
    if max_modes:
        myReof_ds_mode_values = reof_ds.mode.values[0:max_modes]
    else: 
        myReof_ds_mode_values = reof_ds.mode.values

    # For each mode, for each degree, iterate 10 times through random 70% of entries to find a score that passes the NSE
    # for nb_mod in reof_ds.mode.values:
    for nb_mod in myReof_ds_mode_values: 

            # Initialize storage lists
            best_score_list = []
            best_mode_list = []
            best_poly_list = []
            best_deg_list = []

            # Iterate through every degree
            for deg in degrees:

                # Initialize the scores
                score = 0
                best_deg = 0
                best_mode = 0
                best_poly = 0

                # Run 10 times for each degree, 50% of the entries are randomly selected for training
                for iterator in range(150):

                    # Generate all indices
                    all_indices = np.arange(len(reof_ds.time))
                    
                    # Shuffle the indices
                    np.random.shuffle(all_indices)
                    
                    # Determine the split point
                    split_point = int(len(reof_ds.time) * 0.5)  # 70% for training
                    
                    # Create training and test indices
                    inds_train = all_indices[:split_point]
                    inds_test = all_indices[split_point:]

                    # Create the polynomial fit
                    f = np.poly1d(np.polyfit(q[inds_train],reof_ds.temporal_modes.sel(mode=nb_mod)[inds_train],deg=deg))
                    
                    # Predict the values
                    y_pred = f(q[inds_test])

                    # Get the true values
                    Y = reof_ds.temporal_modes.sel(mode=nb_mod)[inds_test]

                    # Calculate NSE. If it passes then the score is compared to the previous best score.
                    nse = 1 - ( (np.nansum(np.square( Y - y_pred ))) / (np.nansum(np.square( Y - np.nanmean(Y) ))) )
                    
                    if nse >= NSE_cut:
                        if nse >= score:
                            score = nse
                            best_deg = deg
                            best_mode = nb_mod
                            best_poly = f.coeffs
                
                # Print the scores that passed the NSE or have the best score
                if score >= NSE_cut:
                    print(f"Mode {nb_mod} - Degree {deg} - NSE {score}")

                # Append the arrays with the best score
                best_score_list.append(score)
                best_deg_list.append(best_deg)
                best_mode_list.append(best_mode)
                best_poly_list.append(best_poly)
            
            # Store best indices passing the NSE
            indmax = np.argmax(best_score_list)
            tot_score.append(best_score_list[indmax])
            tot_deg.append(best_deg_list[indmax])
            tot_mode.append(best_mode_list[indmax])
            tot_poly.append(best_poly_list[indmax])

    # Get 1 entry per mode
    ind_valid = list(np.where(np.array(tot_score)>NSE_cut)[0])
    final_mode = [tot_mode[i] for i in ind_valid]
    temp_f = [tot_poly[i] for i in ind_valid]
    final_deg = [tot_deg[i] for i in ind_valid]
    final_f = []
    inds_final_pass = []

    for ind in range(len(final_mode)):
    
        # Plot the best combination
        f = np.poly1d(temp_f[ind])
        final_f.append(f)
        Y = reof_ds.temporal_modes.sel(mode=final_mode[ind])
        Y_mdl = f(q.values)
        
        # Calculate NSE one more time with all the data
        nse = 1 - ( (np.nansum(np.square( Y - Y_mdl ))) / (np.nansum(np.square( Y - np.nanmean(Y) ))) )

        # Plot if NSE passes
        if nse >= NSE_cut:
            if figSize:
                myFig = plt.figure(figsize=figSize)
            else:
                myFigx = plt.figure()
            plt.scatter(q,reof_ds.temporal_modes.sel(mode=final_mode[ind]), label='original data')
            plt.plot(np.sort(q), f(np.sort(q)), color = 'red', label = 'polyfit')
            plt.legend()
            # set axis limits
            if myAxes:
                plt.xlim(myAxes[0][0], myAxes[0][1])
                plt.ylim(myAxes[1][0], myAxes[1][1])

            # Calculate plot limits
            xlim = plt.xlim()
            ylim = plt.ylim()
            
            # Add text with coordinates relative to the plot limits
            plt.text(
                xlim[0] + 0.8 * (xlim[1] - xlim[0]),  # x-coordinate
                ylim[0] + 0.5 * (ylim[1] - ylim[0]),  # y-coordinate
                'NSE: ' + "{:.2f}".format(nse),
                fontsize=12, color='black', weight='bold'
            )

            # Add date next to each point
            for some_q, some_t in zip(q, reof_ds.temporal_modes.sel(mode=final_mode[ind])):
                plt.annotate(some_q.time.dt.strftime("%Y-%m-%d").values, xy=(some_q,some_t))

            plt.title(f"Mode {final_mode[ind]}")
            
            # force plot to draw now
            plt.draw()
            plt.pause(0.1)

            # save plot
            if myParent and (s_f or s_f==0):
                flnm = f"smoothing-{s_f:02d}_mode-{final_mode[ind]:02d}.png"
                saveLoc = Path(myParent, 'Figures', 'fittingModes', 'poly', f"smoothing-{s_f:02d}")
                # create save location if necessary
                os.makedirs(saveLoc, exist_ok=True)
                myFig.savefig(Path(saveLoc,flnm))
            

            best_nse = nse
            best_score.append(best_nse)
        
        # Store the indices not passing
        else:
            inds_final_pass.append(ind)
        
    # Remove indices flagged as not passing NSE
    final_f = [item for idx, item in enumerate(final_f) if idx not in inds_final_pass]
    final_deg = [item for idx, item in enumerate(final_deg) if idx not in inds_final_pass]
    final_mode = [item for idx, item in enumerate(final_mode) if idx not in inds_final_pass]

    
    return (final_mode, final_deg, final_f, best_score)





### --- Neural Network Initialization --- ###
# ----- Define a callback function, setting up criteria for training to stop -----
callback = tf.keras.callbacks.EarlyStopping(monitor='loss', min_delta=0.000001, patience=10)


# ----- Declare Tensorflow normalizer of data -----
# -- Initialize normalization process (which axis to be along with, and the shape of input) --
# -- "input_shape=(1,)" means only one variable is used as input. 
#    Since the number of observations in a variable can be dynamic, it often remains unprovided, hence defined as (1,)
#    
#    In the case of multivariate regression, says N variables, are used as input, it becomes
#    "input_shape=(N,)"
#    
#    "axis=-1", means the mean and STD calculated in ".adapt(DATA)" is along the last dimension, which in
#    this case is along column. It depends on which axis of data represent the variable types.
normalizer = Normalization(input_shape=(1,),axis=-1) 


### --- Design Tensorflow model structure --- ###
def build_and_compile_model(norm):
    model = keras.Sequential([
        norm,
        layers.Dense(128, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(128, activation='relu'),
        #layers.Dense(128, activation='relu'),
        #layers.Dense(64, activation='relu'),
        layers.Dense(1)               
    ])
    
    model.compile(loss='mean_squared_error',
                  optimizer=tf.keras.optimizers.Adam())#tf.keras.optimizers.Adam(0.001))
    return model# ----- Function for plotting training progress -----





### --- Function for plotting training progress ---###
def plot_loss(history):

    fig = plt.figure()
    ax = fig.add_axes([0,0,1,1])
    plt.plot(history.history['loss'], label='loss')
    #plt.plot(history.history['val_loss'], label='val_loss')  
    plt.xlabel('Epoch')
    plt.ylabel('[Loss] MSE: RTPC')
    #plt.legend()
    plt.text(0.6,0.7,'Final loss: '+"{:.2f}".format(history.history['loss'][-1]),transform=ax.transAxes)
    plt.grid(True)
    plt.savefig('train_loss.png', dpi=300, bbox_inches='tight')    
    plt.show()





### --- Calculate Neural Network fit for modes --- ###
def nn_fit(reof_ds, q, NSE_cut, myParent=None, s_f=None, figSize = None, max_modes=None, myAxes=None):
    indx_qual_mode = []
    best_model = []
    best_score = []
    best_nse = 0

    # select modes to use
    if max_modes:
        myReof_ds_mode_values = reof_ds.mode.values[0:max_modes]
    else: 
        myReof_ds_mode_values = reof_ds.mode.values
    
    # ----- Train Tensorflow Hydro-to-TPC models mode-by-mode -----
    for mode in myReof_ds_mode_values:  
        keras.backend.clear_session()    
        print('Building models for mode-'+str(mode).zfill(2))
        tpc = reof_ds.temporal_modes.sel(mode=int(mode))        

        X = q.values.reshape(len(q),1)
        Y = tpc.values.reshape(len(q),1)
    
        X_train = X
        Y_train = Y    
    
        # -- adapt function is to get the mean and STD used to normalize the input data of the model --
        normalizer.adapt(X_train)    
    
        # ----- Construct the model and get summary -----
        model = build_and_compile_model(normalizer)
        #if vis_tf_nn==0:
        #elif vis_tf_nn==1:
        #  plot_model(model, to_file='NOAA_WF\\hydro2rtpc_mdl\\'+data_src+'\\site-'+str(gaugeID_list[site])+'_tpc'+str(mode+1).zfill(2)+'.png', show_shapes=True, show_layer_names=True)
    
         # ----- Fit the model -----

        # Added by knicely
        early_stopping = tf.keras.callbacks.EarlyStopping(
            monitor='loss', # metric to be monitored, typically 'loss' for training loss or 'val_loss' for validation loss. 
            patience=10, # number of epochs with no improvement after which training will be stopped
        )

        train_proc = model.fit(
            X_train, 
            Y_train, 
            callbacks=[early_stopping],
            batch_size=32, 
            epochs=200, 
            verbose=0, 
            #validation_split=0.2
        )    
        # End addition by Knicely

        # # Original train_proc!!!!
        # train_proc = model.fit(
        #     X_train, 
        #     Y_train, 
        #     callbacks=[callback],
        #     batch_size=32, 
        #     epochs=200, 
        #     verbose=0, 
        #     #validation_split=0.2
        # )    

        # ----- Plot model estimation and original scatter plot -----
        X_sim = tf.linspace(np.amin(X), np.amax(X), X.size*10^10)
        Y_sim = model.predict(X_sim)
   

        # ----- The second-time REOF mode screening based on quality of regression models. If qualified, export trained model -----
        Y_mdl = model.predict(X[:,0])
        nse = 1 - ( (np.nansum(np.square( Y - Y_mdl ))) / (np.nansum(np.square( Y - np.nanmean(Y) ))) )
        print(f'NSE = {nse}')

        if nse >= NSE_cut: # Moriasi et al., 2007. Consider NSE>0.5 as satisfactory
            
            model.summary()
            print(mode, nse)

            # ----- Plot training progress -----  
            plot_loss(train_proc)

            best_nse = nse
            best_score.append(best_nse)
            best_model.append(model)
            indx_qual_mode.append(mode)

            if figSize:
                myFig = plt.figure(figsize=figSize)
            else:
                myFig = plt.figure()
            # set axis limits:
            if myAxes:
                plt.xlim(myAxes[0][0], myAxes[0][1])
                plt.ylim(myAxes[1][0], myAxes[1][1])
            ax = fig.add_axes([0,0,1,1])
            plt.scatter(X, Y)
            plt.plot(X_sim, Y_sim, color='r')
            plt.xlabel('discharge (m3/d)')
            plt.ylabel(f"mode {mode}")  
            plt.xticks(rotation=45, ha='right')
            plt.yticks(rotation=45)
            plt.legend(['NN-Model','Data'],loc='upper left')
            plt.text(0.2,0.5,'NSE: '+"{:.2f}".format(nse), fontweight='bold', transform=ax.transAxes)

            # Add date next to each point
            for some_q, some_t in zip(q, reof_ds.temporal_modes.sel(mode=int(mode))):
                plt.annotate(some_q.time.dt.strftime("%Y-%m-%d").values, xy=(some_q,some_t))
            
            plt.show()

            # save plot
            if myParent and (s_f or s_f==0):
                flnm = f"smoothing-{s_f:02d}_mode-{mode:02d}.png"
                saveLoc = Path(myParent, 'Figures', 'fittingModes', 'neural-network', f"smoothing-{s_f:02d}")
                # create save location if necessary
                os.makedirs(saveLoc, exist_ok=True)
                myFig.savefig(Path(saveLoc,flnm))
        

    
            # ----- Export trained model -----
            #best_model.save('test.keras')  
    # ADDED by Knicely
        del tpc, X, Y, X_train, Y_train, model, train_proc, X_sim, Y_sim, Y_mdl, nse
    keras.backend.clear_session()    
    
    return(best_model, indx_qual_mode, best_score)

## SELECT DATA FOLDER

In [ ]:
from ipyfilechooser import FileChooser
fc = FileChooser(Path.cwd())
display(fc)

In [ ]:
fc.selected

***
## LOAD DATA

Select the input file type: water masks ('Water_Masks') or SAR imagery ('SAR')

In [ ]:
# Input type of file: 'Water_Masks' or 'SAR
type_file = 'Water_Masks'

# Input type of polarization for SAR: 'VV' or 'VH'
if type_file == 'SAR':
    pola = 'VV'
else:
    pola = 'combined'
import time

## Load data for plotting and later input into the REOF. 

In [ ]:
# Import flood percentage calculation from water masks
floodpercent = np.load(fc.selected+"Figures/flood_percentage.npy")

# Import time associated with the floods
time_flood = np.load(fc.selected+"Figures/time_index.npy")

### --- Choose an index that represents a flood --- ###
ind_flood = 103 # for subset_bang; second to last flood
# ind_flood = 138 # for subset_bang; last flood
# ind_flood = 69 # for subset-Nepal
ind_flood = 192 # for sylhet_retry/central

# Plot the flood percentage and index chosen
fig_flood = plt.figure()
plt.scatter(time_flood, floodpercent, label = 'Flood percentage in time')
plt.scatter(time_flood[ind_flood],floodpercent[ind_flood],color='red', label = 'Chosen flood')
plt.title('Flood percentage and chosen flood')
plt.ylabel('Flood percentage')
plt.xticks(rotation=45)
plt.grid()
plt.legend()

plt.show()

### Load and filter geoglows data

We will load our image stack, screen them to the relevant time frame, and then get the corresponding discharge forecast from Geoglows. 

In [ ]:
# Load tiffs
da_tot, tiffs_tot, times_tot = load_tiffs(fc.selected, type_file, pola, stop_ind = -1)

# Check match between selected date of flood and tiffs
if da_tot.time[ind_flood].values == time_flood[ind_flood]:
    print('Dates of floods and tiffs are matching')
else:
    print('Dates of floods and tiffs are not matching')

# Grab lat/lon of tiffs to get discharge forecast
lat, lon = get_centerpoint_coordinates(tiffs_tot[0])


In [ ]:
# lat, lon = 28.8722,80.6621 # manual selection of lat/lon for Nepal site

In [ ]:
# Get discharge forecast
print('Getting streamflow data.')
# q_tot, ID = get_streamflow(lat, lon)
q_tot, ID = g_sf_new(lat,lon)
print('Streamflow data acquired.')

### Load and filter other data (IN PROGRESS)

River gauge???



### Temporal Smoothing of Data (OPTIONAL)

Apply a temporal smoothing filter to the data to which to correlate the SAR-derived flood masks. 

Select your desired smoothing range in days. 

Note: A smoothing frame of 1 day technically does nothing as the input GEOGLOWS data already is set to daily rather than in some increment. 

TO DO: 
* Add code to identify the length of time between entries in data and convert smoothing_frame to that. 

In [ ]:
### --- Smoothing function --- ###

# I AM HERE; this smoothing function needs to be tested! 
# The current form is basically a moving average filter, which ends up retroactively smoothing data. This will be a problem when it comes to doing true forecasts; won't be able to back-propagate, only forward smooth. 
def smooth(y, box_pts, forwardOnly = False):
    
    if forwardOnly:
        # only smooth forward in time
        box = np.zeros(box_pts)
        myIdx = np.round(box_pts/2.0)
        box[myIdx:] = 1 / len(box[myIdx:])        
    else:
        # do a moving average filter (which back-propagates smoothing)
        box = np.ones(box_pts)/box_pts

    
    y_smooth = np.convolve(y, box, mode='same')
    
    return y_smooth

In [ ]:
# Set the number of days over which to smooth data
smoothing_frame = [1] # this must be a list (even if just a single entry)! Number of days over which to smooth. 
smoothing_frame = [3, 19]
smoothing_frame = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40]

# make sure there is a leading zero in the 'smoothing_frame' list; this is important for loops later on. 
if smoothing_frame[0] != 0:
    smoothing_frame.insert(0,0)
smoothing_frame

In [ ]:
q_tot_smoothed = q_tot.copy(deep=True)

if smoothing_frame: # if 'smoothing_frame' isn't empty, then do all this stuff
    q_new = q_tot.copy(deep=True)
    
    for i in range(1,len(smoothing_frame)): # skip the first entry of 'smoothing_frame' (which should just be the original, without smoothing!)
        temp = smooth(q_tot.values, smoothing_frame[i])
        q_new.values = smooth(q_tot.values, smoothing_frame[i])
        q_tot_smoothed = xr.concat([q_tot_smoothed, q_new], dim="discharge")

### Plot flood and discharge data over full time series

In [ ]:
def plot_fpONdis(t_f, fp, i_f, t_t, q_t, smoothing=[]):
    # t_f = time_flood
    # fp = floodpercent
    # i_f = ind_flood
    # t_t = times_tot
    # q_t = q_tot
    fig_fNd, ax_flood = plt.subplots(figsize=(12,6))
    # Plot the flood percentage and index chosen
    lns1 = ax_flood.plot(t_f, fp, color='blue', linestyle= 'None', marker='o', label = 'Flood percentage in time')
    lns2 = ax_flood.plot(t_f[i_f],fp[i_f],color='red', linestyle= 'None', marker='x', markersize=20, label = 'Chosen flood')
    
    ax_disch = ax_flood.twinx() # twin the axis so both can be on the same plot. 
    # get matching date for dischage and images/masks
    idxA, idxB = np.where(t_t[0] == q_t.time)[0][0], np.where(t_t[-1] == q_t.time)[0][0]
    lns3 = ax_disch.plot(q_t[idxA:idxB].time,q_t[idxA:idxB], color='green', label='Discharge')
    
    # set axis
    if smoothing or smoothing==0:
        ax_flood.set_title(f'Flood percentage and chosen flood\nSmoothing Frame = {smoothing} days')
    else:
        ax_flood.set_title('Flood percentage and chosen flood')
    ax_flood.set_ylabel('Flood percentage')
    ax_disch.set_ylabel('Discharge [$m^3/s$]')
    ax_flood.grid()
    
    # combine the line labels into one for the legend. 
    lns = lns1+lns2+lns3
    labs = [l.get_label() for l in lns]
    ax_flood.legend(lns, labs)
    
    import matplotlib.dates as mdates
    ax_flood.xaxis.set_major_locator(mdates.MonthLocator(bymonth=range(1,12,3)))
    ax_flood.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    
    
    
    plt.setp(ax_flood.get_xticklabels(), rotation=45)
    
    plt.show()

In [ ]:
try:
    for i in range(0,len(smoothing_frame)):
        plot_fpONdis(time_flood, floodpercent, ind_flood, times_tot, q_tot_smoothed[i], smoothing_frame[i])
except: 
    plot_fpONdis(time_flood, floodpercent, ind_flood, times_tot, q_tot)

## Smoothing check

The following code provides a visualization and a simple root means square difference (RMSD) metric to verify that the smoothing frame selected has actually resulted in a change in the discharge. 

In [ ]:
def plot_fpONdis_smoothed(t_f, fp, i_f, t_t, q_t, s_f = [], matching = None):
    # t_f = time_flood
    # fp = floodpercent
    # i_f = ind_flood
    # t_t = times_tot
    # q_t = q_tot
    # s_f = smoothing_frame
    fig_fNd, ax_flood = plt.subplots(figsize=(12,6))
    # Plot the flood percentage and index chosen
    lns1 = ax_flood.plot(t_f, fp, color='blue', linestyle= 'None', marker='o', label = 'Flood percentage in time')
    lns2 = ax_flood.plot(t_f[i_f],fp[i_f],color='red', linestyle= 'None', marker='x', markersize=20, label = 'Chosen flood')
    
    ax_disch = ax_flood.twinx() # twin the axis so both can be on the same plot. 
    # get matching date for dischage and images/masks

    from matplotlib import cm
    myColors = cm.rainbow(np.linspace(0,1,len(s_f)))
    
    lns = []
    if matching:
        # find matching times
        idxs = []
        for i in range(0,len(t_t)): 
            idx = np.where(t_t[i] == q_t.time)[0][0]
            idxs.append(idx)
        # plot as a scatter plot
        # lns3 = ax_disch.scatter(q_t[idxs].time, q_t[idxs], color='green', label='Discharge')
        for i in range(0,len(s_f)):
            # lns3 = ax_disch.plot(q_t[i,idxs].time, q_t[i,idxs], color='green', label='Discharge', marker = 'o', linestyle='None')
            myLabel = f'Smoothing Frame - {s_f[i]} days'
            ax_disch.plot(q_t[i,idxs].time, q_t[i,idxs], color=myColors[i], label=myLabel, marker = 'o', linestyle='None')
    else:
        idxA, idxB = np.where(t_t[0] == q_t.time)[0][0], np.where(t_t[-1] == q_t.time)[0][0]
        for i in range(0,len(s_f)):
            # lns3[i] = ax_disch.plot(q_t[i,idxA:idxB].time, q_t[i,idxA:idxB], color='green', label='Discharge')
            myLabel = f'Smoothing Frame - {s_f[i]} days'
            ax_disch.plot(q_t[i,idxA:idxB].time, q_t[i,idxA:idxB], color=myColors[i], label=myLabel)
    
    # set axis
    if s_f:
        ax_flood.set_title(f'Flood percentage and chosen flood\nSmoothing Frame = {s_f} days')
    else:
        ax_flood.set_title('Flood percentage and chosen flood')
    ax_flood.set_ylabel('Flood percentage')
    ax_disch.set_ylabel('Discharge [$m^3/s$]')
    ax_flood.grid()
    
    # combine the line labels into one for the legend. 
    lns = lns1+lns2
    labs = [l.get_label() for l in lns]
    ax_flood.legend(lns, labs)
    
    import matplotlib.dates as mdates
    ax_flood.xaxis.set_major_locator(mdates.MonthLocator(bymonth=range(1,12,3)))
    ax_flood.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    
    plt.legend()
    
    plt.setp(ax_flood.get_xticklabels(), rotation=45)
    
    plt.show()

In [ ]:

plot_fpONdis_smoothed(time_flood, floodpercent, ind_flood, times_tot, q_tot_smoothed, smoothing_frame, matching=False)

In [ ]:
plot_fpONdis_smoothed(time_flood, floodpercent, ind_flood, times_tot, q_tot_smoothed[0:len(q_tot_smoothed):5], smoothing_frame[0:len(q_tot_smoothed):5], matching=False)

In [ ]:
plot_fpONdis_smoothed(time_flood, floodpercent, ind_flood, times_tot, q_tot_smoothed, smoothing_frame, matching=True)

In [ ]:
plot_fpONdis_smoothed(time_flood, floodpercent, ind_flood, times_tot, q_tot_smoothed[0:len(q_tot_smoothed):5], smoothing_frame[0:len(q_tot_smoothed):5], matching=True)

In [ ]:
# Calculate root mean difference for matched dates that are smoothed
# find matching times
idxs = []
for i in range(0,len(times_tot)): 
    idx = np.where(times_tot[i] == q_tot_smoothed.time)[0][0]
    idxs.append(idx)

# loop through each smoothing frame and calucalte root mean square difference from the first
myRMSD = np.zeros(len(smoothing_frame))
for i in range(0,len(smoothing_frame)): 
    myRMSD[i] = np.sqrt(np.mean((q_tot_smoothed[0,idxs].values - q_tot_smoothed[i,idxs].values)**2))
    print(f'RMSD for {smoothing_frame[i]} days of smoothing: {myRMSD[i]}')

plt.bar(smoothing_frame, myRMSD)
plt.xlabel('Days of Smoothing')
plt.ylabel('RMSD')
plt.title('RMSD for various Smoothing Frames')
plt.grid()
plt.show()

### Plot imagery (water masks or SAR based on earlier selection) of pre- and during flooding, their difference, and the relevant discharge data over that time. 

In [ ]:
### --- visual flood analysis plots --- ###
# This shows the pre-flooding, during flooding, the difference between the two, and the relevant discharge data over that time. 
def plot_floodComparison(d_t, i_f, t_f, q_t, smoothing=[]):
    # d_t = da_tot
    # i_f = ind_flood
    # t_f = type_file
    # q_t = q_tot

    # get matching date for discharge and images/masks
    idx1, idx2 = np.where(d_t[i_f-1].time == q_t.time)[0][0], np.where(d_t[i_f].time == q_t.time)[0][0]
    
    # plot of pre (ind_flood-1) and during (ind_flood) flooding from water_masks/SAR images. 
    fig_flooding = plt.figure(figsize=(9,9))
    ax_pre = plt.subplot(2,3,1)
    ax_during = plt.subplot(2,3,2)
    ax_diff = plt.subplot(2,3,3)
    if t_f == 'Water_Masks':
        vmin, vmax = 0, 1
    elif t_f == 'SAR':
        vmin, vmax = 0, 255
    ax_pre.imshow(d_t[i_f-1], vmin=vmin, vmax=vmax, cmap='seismic_r', interpolation=None) 
    ax_during.imshow(d_t[i_f], vmin=vmin, vmax=vmax, cmap='seismic_r', interpolation=None)
    d_diff = d_t[i_f] - d_t[i_f-1]
    ax_diff.imshow(d_diff, vmin=vmin, vmax=vmax, cmap='seismic_r', interpolation=None)
    ax_pre.title.set_text(f'Pre-flooding\n{q_t[idx1].time.dt.strftime("%Y-%m-%d").values}')
    ax_during.title.set_text(f'During Flooding\n{q_t[idx2].time.dt.strftime("%Y-%m-%d").values}')
    ax_diff.title.set_text('Difference')    
    if smoothing or smoothing==0:
        fig_flooding.suptitle(f'Smoothing Frame = {smoothing} days')
                                           
    # plot of discharge through time
    ax_dis = plt.subplot(2,1,2)
    xtra = 5
    ax_dis.plot(q_t[idx1-xtra:idx2+xtra].time,q_t[idx1-xtra:idx2+xtra])
    ax_dis.plot(q_t[idx1].time, q_t[idx1], color='green', marker='x', markersize=10)
    ax_dis.plot(q_t[idx2].time, q_t[idx2], color='green', marker='x', markersize=10)
    ax_dis.grid()
    ax_dis.title.set_text('Discharge vs Time')
    ax_dis.set_ylabel('Discharge [$m^3/s$]')
    plt.xticks(rotation=45)
    
    plt.show()

In [ ]:
try:
    for i in range(0,len(smoothing_frame)):
        plot_floodComparison(da_tot, ind_flood, type_file, q_tot_smoothed[i], smoothing_frame[i])
except:
    plot_floodComparison(da_tot, ind_flood, type_file, q_tot)

## Prep for the REOF

### Parse data to relevant times for the REOF

This takes the data from 'q_tot' and cuts it down to the training time. This training time is from the first day of SAR data/water masks to the flood event we are trying to predict. 

In [ ]:
# Crop tiffs and dates to the dates of interest, keep the rest
da = da_tot[:ind_flood]
tiffs = tiffs_tot[:ind_flood]
times = times_tot[:ind_flood]

# Crop to date range
q_tot = q_tot.resample(time='1D').mean().rolling(time=5).mean()
q = match_dates(q_tot, da.time)

In [ ]:
# crop smoothed data to date range

try: 
    q_smoothed = q.copy(deep=True)
    print(f'1 of {len(smoothing_frame)} cropped...')
    for i in range(1,len(smoothing_frame)):
        q_tot_smoothed[i] = q_tot_smoothed[i].resample(time='1D').mean().rolling(time=5).mean()
        temp = match_dates(q_tot_smoothed[i], da.time)
        q_smoothed = xr.concat([q_smoothed, temp], dim='discharge')
        print(f'{i+1} of {len(smoothing_frame)} cropped...')
except: 
    print('Temporally smoothed data has not been downsampled to daily.')

print('Cropping complete!')

### Temporal Filter (OPTIONAL)
Remove data from non-rainy times

From approximately Nov. 1st to Jan. 1st, there are numerous high flood percentage days with very low discharge. These data reduce the correlation between discharge and flooding. Their removal tends to improve flood forecasting. 

In [ ]:
# select time frame to keep

import datetime
keep_start = datetime.datetime(2020, 5, 1) # year, month, day; the year doesn't actually matter
keep_stop  = datetime.datetime(2020, 11, 30)

print(f"Selected date range to keep: {keep_start.strftime('%m-%d')} to {keep_stop.strftime('%m-%d')}")
if keep_start > keep_stop: 
    print("\n\nWARNING!!!\nWARNING!!!\nWARNING!!!\nStop date is before the start date!")

In [ ]:
# get list of indices to keep
idx_keep = []
for i in range(len(da.time)):
    tempy = pd.Timestamp(da[i].time.values)
    if (tempy.month, tempy.day) > (keep_start.month, keep_start.day) and (tempy.month, tempy.day) < (keep_stop.month, keep_stop.day):
        idx_keep.append(i)

In [ ]:
# Crop tiffs and dates to the dates of interest, keep the rest
da = da_tot[idx_keep]
# tiffs_tot requires a special methodology for some reason
tiffs = []
for idx in idx_keep:
    tiffs.append(tiffs_tot[idx])
tiffs
times = times_tot[idx_keep]

# Crop to date range
q_tot = q_tot.resample(time='1D').mean()
q = match_dates(q_tot, da.time)

In [ ]:
# copy over data for times we want to keep

try: 
    q_smoothed = q.copy(deep=True)
    print(f'1 of {len(smoothing_frame)} filtered...')
    for i in range(1,len(smoothing_frame)):
        q_tot_smoothed[i] = q_tot_smoothed[i].resample(time='1D').mean().rolling(time=5).mean()
        temp = match_dates(q_tot_smoothed[i], da.time)
        q_smoothed = xr.concat([q_smoothed, temp], dim='discharge')
        print(f'{i+1} of {len(smoothing_frame)} filtered...')
except: 
    print('Temporally smoothed data has not been downsampled to daily.')

print('Time filtering complete!')

## Run the REOF

In [ ]:
# Calculate REOF of the tiffs
myTolerance = 1e-5
# Number of temporal modes to use; technically, this should be the number of images (or water masks) that you have. 
n_modes = 10 # fast run time
n_modes = len(da) # proper REOF
# Run the REOF calculation. 
reof_ds = reof(da, n_modes, myTolerance)

### Visualize the temporal modes

In [ ]:
num_modes = n_modes
num_modes = 10

# Create subplots: 2 rows, 5 columns
fig, axes = plt.subplots(2, 5, figsize=(15, 9))

# Flatten axes array for easier indexing
axes = axes.flatten()

# Loop through the first 10 modes and plot
for i in range(num_modes):
    ax = axes[i]
    ax.scatter(reof_ds.time, reof_ds.temporal_modes[i], label=f'Mode {i+1}', s=10)
    ax.set_title(f'Temporal Mode {i+1}')
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('Time')
    ax.set_ylabel('Amplitude')
    ax.grid()
    ax.set_box_aspect(1)

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

In [ ]:
# Create subplots: 2 rows, 5 columns
fig, axes = plt.subplots(5, 2, figsize=(15, 40))

# Flatten axes array for easier indexing
axes = axes.flatten()

# Loop through the first 10 modes and plot
for i in range(num_modes):
    ax = axes[i]
    ax.plot(reof_ds.time, reof_ds.temporal_modes[i], label=f'Mode {i+1}')
    ax.set_title(f'Temporal Mode {i+1}')
    ax.tick_params(axis='x', rotation=45)
    ax.set_xlabel('Time')
    ax.set_ylabel('Amplitude')
    ax.grid()
    # ax.set_box_aspect(1)

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

In [ ]:
# Create subplots: 2 rows, 5 columns
fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=True)

# Flatten axes array for easier indexing
axes = axes.flatten()

# Loop through the first 10 modes and plot
for i in range(num_modes):
    ax = axes[i]
    ax.scatter(q.values, reof_ds.temporal_modes[i], label=f'Mode {i+1}', s=10)
    ax.set_title(f'Temporal Mode {i+1}')
    ax.set_xlabel('Discharge [$m^3/s$]')
    ax.set_ylabel('Amplitude')
    ax.grid()

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

### Visualize Spatial Modes

In [ ]:
import matplotlib.colors as colors

# Create subplots: 2 rows, 5 columns
fig, axes = plt.subplots(2, 5, figsize=(15, 20), sharex=True)

# Flatten axes array for easier indexing
axes = axes.flatten()

# cb_min, cb_max = np.amin(reof_ds.spatial_modes.values), np.amax(reof_ds.spatial_modes.values) # original
cb_min, cb_max = np.amin(reof_ds.spatial_modes.values[0:num_modes, :, :]), np.amax(reof_ds.spatial_modes.values[0:num_modes, :, :]) # changed to avoid poor display of modes (most are very close to middle values, not the maxs/mins)

# Loop through the first 10 modes and plot
for i in range(num_modes):
    ax = axes[i]
    spatial_mode = reof_ds.spatial_modes.values[i,:,:]
    norm = colors.TwoSlopeNorm(vmin=cb_min, vcenter = 0, vmax=cb_max)
    im = ax.imshow(spatial_mode, label=f'Mode {i+1}', cmap='seismic_r', norm=norm, origin='lower')
    ax.set_title(f'Spatial Mode {i+1}')

# create horizontal colorbar


# Adjust layout and show the plot
plt.tight_layout()
fig.colorbar(im, ax=axes, location='bottom', orientation='horizontal')
plt.show()

## Find best fits polynomial and NN

- Choose the maximum amount of polynomial degrees to evaluate for the fit
- Choose the NSE-score cutoff used to keep polynomial fits as valuable

In [ ]:
### --- CHOOSE THE MAX DEGREE OF THE POLYNOMIALS TO EVALUATE --- ###
num_deg = 5

### --- CHOOSE THE NSE CUT-OFF FOR NSE SCORE OF THE POLYNOMIALS --- ###
NSE_cut = 0.1
NSE_cut = 0.01 # I AM HERE; get fits for every mode

### --- SET THE AXES LIMITS (optional) --- ###
x0, x1 = np.amin(q_smoothed.values), np.amax(q_smoothed.values)
x_diffy = np.abs( x1-x0 ) / 10.0
y0, y1 = np.amin(reof_ds.temporal_modes.values), np.amax(reof_ds.temporal_modes.values)
y_diffy = np.abs( y1-y0 ) / 10.0
myAxes = [[x0-x_diffy, x1+x_diffy],
          [y0-y_diffy, y1+y_diffy]]

### --- Loop through each entry of 'smoothing_frame' to get polynomial fits --- ###

# Initialize containers for output results
modes_poly_smoothed = []
degs_smoothed       = []
polys_smoothed      = []
NSEs_poly_smoothed  = []

try: 
    for i in range(0,len(smoothing_frame)):
        print(f'Calculting fit {i+1} of {len(smoothing_frame)}...')
        modes_poly_s, degs_s, polys_s, NSEs_poly_s = best_polynomial_fits(reof_ds, q_smoothed[i], num_deg, NSE_cut, fc.selected, smoothing_frame[i], figSize=[20,20], max_modes=num_modes, myAxes=myAxes)
        modes_poly_smoothed.append(modes_poly_s)
        degs_smoothed.append(degs_s)
        polys_smoothed.append(polys_s)
        NSEs_poly_smoothed.append(NSEs_poly_s)
        if modes_poly_s == []:
            print(f'No polynomial fits were found with NSE >= {NSE_cut}\nSmoothing_frame = {smoothing_frame[i]}\n')
        else:
            print(f'Polynomial fits found for modes: {modes_poly_s}\ndegrees: {degs_s}\nSmoothing_frame = {smoothing_frame[i]}\n')
except:
    print('Loop over each entry of \'smoothing_frame\' did not execute!!!')

In [ ]:
### --- Loop through each entry of 'smoothing_frame' to get neural network fits --- ###

try:
    del models_nn_smoothed, modes_nn_smoothed, NSEs_nn_smoothed, models_nn_s, modes_nn_s, NSEs_nn_s
except:
    pass

# Initialize containers for output results
models_nn_smoothed = []
modes_nn_smoothed  = []
NSEs_nn_smoothed   = []

try: 
    for i in range(0,len(smoothing_frame)):
        print(f'Calculting fit {i+1} of {len(smoothing_frame)}...')
        models_nn_s, modes_nn_s, NSEs_nn_s = nn_fit(reof_ds, q_smoothed[i], NSE_cut, myParent=fc.selected, s_f=smoothing_frame[i], figSize=[20,20], max_modes=num_modes, myAxes=myAxes)
        models_nn_smoothed.append(models_nn_s)
        modes_nn_smoothed.append(modes_nn_s)
        NSEs_nn_smoothed.append(NSEs_nn_s)
        if modes_nn_s == []:
            print(f'No neural network fits were found with NSE >= {NSE_cut}\nSmoothing_frame = {smoothing_frame[i]}\n')
        else:
            print(f'Neural network fits found for modes: {modes_nn_s}\nmodels: {models_nn_s}\nSmoothing_frame = {smoothing_frame[i]}\n')
except:
    print('Loop over each entry of \'smoothing_frame\' did not execute!!!')

***
## Identify the Best Filtering Set

This section of code identifies which set of smoothing provides the best data fit. This is determined via a weighted summation of the R-squared value multiplied by the inverse of the mode and then divided by the summation of the inverse modes. This is shown in the equation below. 

$$
Score = \frac{\sum_{i=1}^N R^2 / i}{\sum_{i=1}^N 1/i}
$$


N is the total number of modes used. i is the ith mode. 

This particular method was chosen as the higher the mode, the less data it explains. Therefore, higher modes are less desirable. The denominator was chosen so that the maximum score returned is 1. 
The code has been written with a multi-mode version in mind. 

In [ ]:
### --- Visualize fitting results --- ###

def plot_metrics(modes, r2s):
    xs = np.arange(0,len(modes))
    plt.bar(xs, r2s)
    plt.xticks(ticks=xs, labels=modes)
    plt.ylim(0,1)
    plt.xlabel('Mode')
    plt.ylabel('Score (r^2)')
    plt.show()

def plot_metrics_smoothed(scores_nn, scores_poly, s_f):
    # s_f = smoothing_frame
    width = 0.25
    xs = np.arange(0,len(s_f))
    plt.bar(xs, scores_nn, label='Neural Network', width=width)
    plt.bar(xs+width, scores_poly, label='Polynomial', width=width)
    plt.xticks(ticks=xs, labels=s_f)
    plt.ylim(0,1)
    plt.title('R Squared Value for Various Smoothings')
    plt.xlabel('Days of Smoothing')
    plt.ylabel('Score')
    plt.legend()
    plt.show()

In [ ]:
for i in range(0,len(modes_poly_smoothed)):
    print(f'Polynomial\nSmoothing_frame = {smoothing_frame[i]}')
    plot_metrics(modes_poly_smoothed[i], NSEs_poly_smoothed[i])

In [ ]:
for i in range(0,len(modes_nn_smoothed)):
    print(f'Neural Network\nSmoothing_frame = {smoothing_frame[i]}')
    plot_metrics(modes_nn_smoothed[i], NSEs_nn_smoothed[i])

In [ ]:
### --- define function to calculate a normalized score for fitting --- ###
def score(modes, r2s, max_modes=[]):
    # modes     - the list of modes for which a sufficiently good line was fitted
    # r2s       - the r squared value for each mode in modes
    # max_modes - the maximum number of modes to consider for calculating the score; this is generally kept the same as the number of modes for which a polynomial/neural network fit were determined (usually 10)

    # temp variables created to prevent modification of originals. 
    myModes = []
    missingModes = []
    myR2s = []
    missingR2s = []
    if not max_modes:
        max_modes = 10

    # get modes up to max_modes
    for i in range(0,max_modes): 
        try:
            if modes[i] <= max_modes:
                myModes.append(modes[i])
                myR2s.append(r2s[i])
        except:
            pass
    
    # add 'missing' modes and r2s
    # note: all 'missing' r2s are given values of 0. This is to normalize the comparison to other fits. In other words, this is done to prevent high mode fits from falsely deflating scores. 
    # E.g., if a poly fit has only up to modes 5, but a neural network fit has one at mode 10, all of those extras cause the neural network to have a much lower score, 
    # even though the modes that the neural network and poly fits share are identical. 
    min_val = min(modes)
    max_val = max_modes
    full_range = set(range(min_val, max_val+1))
    input_set = set(modes)
    missingModes = sorted(list(full_range - input_set))
    myModes = myModes + missingModes
    zeros_to_add = [0] * len(missingModes)
    myR2s.extend(zeros_to_add)

    # make 'myR2s' and 'myModes' integer arrays
    myR2s = np.array(myR2s)
    myModes = np.array(myModes)
    
    # # check of added 'missing' modes and r2s
    # for i in range(0, len(myModes)): 
    #     print(f'myMode = {myModes[i]}, myR2s = {myR2s[i]}')
    
    # calculate the weighted score. 
    if max_modes:
        S = np.nansum( np.divide(myR2s[myModes<=max_modes],myModes[myModes<=max_modes]) ) / np.nansum( np.divide(1,myModes[myModes<=max_modes]) )
    else: 
        S = np.nansum( np.divide(myR2s,myModes) ) / np.nansum( np.divide(1,myModes) )
    
    return S

In [ ]:
### --- Calculate scores --- ###
max_modes = 1 # I AM HERE; do this for multiple 'max_modes'

# polynomial
scores_poly = np.zeros(len(smoothing_frame))
for i in range(0,len(smoothing_frame)):
    scores_poly[i] = score( modes_poly_smoothed[i], NSEs_poly_smoothed[i], max_modes ) 

# neural network
scores_nn = np.zeros(len(smoothing_frame))
for i in range(0,len(smoothing_frame)):
    scores_nn[i] = score( modes_nn_smoothed[i], NSEs_nn_smoothed[i], max_modes )

plot_metrics_smoothed(scores_nn, scores_poly, smoothing_frame)

In [ ]:
### --- Calculate scores --- ###
max_modes = 3 # I AM HERE; do this for multiple 'max_modes'

# polynomial
scores_poly = np.zeros(len(smoothing_frame))
for i in range(0,len(smoothing_frame)):
    scores_poly[i] = score( modes_poly_smoothed[i], NSEs_poly_smoothed[i], max_modes ) 

# neural network
scores_nn = np.zeros(len(smoothing_frame))
for i in range(0,len(smoothing_frame)):
    scores_nn[i] = score( modes_nn_smoothed[i], NSEs_nn_smoothed[i], max_modes )

plot_metrics_smoothed(scores_nn, scores_poly, smoothing_frame)

In [ ]:
### --- Calculate scores --- ###
max_modes = 5 # I AM HERE; do this for multiple 'max_modes'

# polynomial
scores_poly = np.zeros(len(smoothing_frame))
for i in range(0,len(smoothing_frame)):
    scores_poly[i] = score( modes_poly_smoothed[i], NSEs_poly_smoothed[i], max_modes ) 

# neural network
scores_nn = np.zeros(len(smoothing_frame))
for i in range(0,len(smoothing_frame)):
    scores_nn[i] = score( modes_nn_smoothed[i], NSEs_nn_smoothed[i], max_modes )

plot_metrics_smoothed(scores_nn, scores_poly, smoothing_frame)

In [ ]:
### --- Calculate scores --- ###
max_modes = 10 # I AM HERE; do this for multiple 'max_modes'

# polynomial
scores_poly = np.zeros(len(smoothing_frame))
for i in range(0,len(smoothing_frame)):
    scores_poly[i] = score( modes_poly_smoothed[i], NSEs_poly_smoothed[i], max_modes ) 

# neural network
scores_nn = np.zeros(len(smoothing_frame))
for i in range(0,len(smoothing_frame)):
    scores_nn[i] = score( modes_nn_smoothed[i], NSEs_nn_smoothed[i], max_modes )

plot_metrics_smoothed(scores_nn, scores_poly, smoothing_frame)

In [ ]:
### --- Select the 'best' model for polynomial and neural network --- ###

# Make selection
# for polynomials
max_poly = np.max(scores_poly)
idx_poly = np.where(scores_poly == max_poly)[0][0]
# for neural network
max_nn = np.max(scores_nn)
idx_nn = np.where(scores_nn == max_nn)[0][0]

# print out if they have the same smoothing or not
if idx_poly != idx_nn:
    print(f'Warning!!!\nBest polynomial and best neural network have different smoothings!\nHighest score has been used to select the best models to return.')
    
    # Select which of the two to use
    if max_nn > max_poly:
        myIdx = idx_nn
    else:
        myIdx = idx_poly
else:
    myIdx = idx_nn

# I AM HERE; assign best models and modes to objects for saving



print(f'\nSelected Index and Scores\nIndex: {myIdx}\nNumber of days smoothed: {smoothing_frame[myIdx]}\nScore(poly, nn): {scores_poly[myIdx]}, {scores_nn[myIdx]}')

In [ ]:
modes_nn_smoothed[myIdx]

In [ ]:
modes_poly_smoothed[myIdx]

***
# Save Models for Forecasting

The polynomial and neural network models are saved here. These are then loaded by the python script which will be executed by the cron job. This is to reduce the amount of processing and computing resources needed. The logic is that the training notebook (this one) will only need to be run possibly once per year; ostensibly, the training data will not need to change very often. Only if there is a very large flooding event (larger than those in the current body of records) will new data need to be incorporated. This is in contrast to the forecasting, which will need to be done as much as every day; ideally, this will only be needed to run once every few days. By splitting the notebooks, the user does not have to do the highly computing intensive REOF every single time they want a new forecast

In [ ]:
from datetime import datetime

In [ ]:
# Load one of the SAR images or water masks in order to get the coordinate reference system. This is used in saving the forecasts as geotiffs. 
if type_file == 'Water_Masks':
    folder = fc.selected+'Water_Masks/'
else:
    folder = fc.selected+'RTC_GAMMA/'

# Gather names of files corresponding to the file type and polarization we want
tiff_dir = Path(folder)
# tiffs = list(tiff_dir.glob(f'{prefix}.tif*')) # for water masks
tiffs = list(tiff_dir.glob(f'*.tif*'))

raster = rasterio.open(tiffs[0])

CRS = raster.crs
CRS

In [ ]:
transform = raster.transform
transform

In [ ]:
# load needed modules
import _pickle as cPickle
from datetime import datetime

In [ ]:
# create the folder in which to store 'my_vz'
# saveFolder = Path(fc.selected, 'Forecasting-Files_'+datetime.today().strftime('%Y-%m-%d'))
saveFolder = Path(fc.selected, 'Forecasting-Files')
saveFolder.mkdir(parents=True, exist_ok=True)

In [ ]:
model_sel = 1 # use the polynomial
# model_sel = 0 # use the neural network

In [ ]:
# use cPickle to store all files in a single file

# NOTE: using this methodology requires they be saved and loaded in the same order. 

saveFile = f'Forecasting-Files_ReachID-{ID}_Created_{datetime.today().strftime('%Y-%m-%d')}'
saveFile = f'smoothingCheck_Forecasting-Files_ReachID-{ID}_Created_{datetime.today().strftime('%Y-%m-%d')}'

start = time.time()

import os
try:
    os.remove(Path(saveFolder,saveFile+".pkl"))
    print("old file deleted")
except OSError:
    pass

with open(Path(saveFolder,saveFile+".pkl"), "wb") as f:
    cPickle.dump(reof_ds.isel(mode=slice(0,max_modes)),f)
    cPickle.dump(modes_poly_smoothed,f)
    cPickle.dump(degs_smoothed,f)
    cPickle.dump(polys_smoothed,f)
    cPickle.dump(models_nn_smoothed,f)
    cPickle.dump(modes_nn_smoothed,f)
    cPickle.dump(q_smoothed,f)
    cPickle.dump(q_tot_smoothed,f)
    cPickle.dump(type_file,f)
    cPickle.dump(da_tot[ind_flood],f)
    cPickle.dump(ind_flood,f)
    cPickle.dump(floodpercent,f)
    cPickle.dump(time_flood,f)
    cPickle.dump(lat,f)
    cPickle.dump(lon,f)
    cPickle.dump(ID,f)
    cPickle.dump(model_sel,f)
    cPickle.dump(CRS,f)
    cPickle.dump(transform,f)
    cPickle.dump(smoothing_frame,f)
    cPickle.dump(scores_nn,f)
    cPickle.dump(scores_poly,f)
    cPickle.dump(NSEs_poly_smoothed,f)
    cPickle.dump(NSEs_nn_smoothed,f)
    cPickle.dump(myIdx,f)

end = time.time()

print('It took ',end-start,' seconds to save the files. ')